In [1]:
# imports
import tensorflow as tf
from tensorflow import keras as keras
import matplotlib.pyplot as plt
import keras_tuner as kt

2024-05-19 10:49:40.621632: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-05-19 10:49:40.621686: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-05-19 10:49:40.733690: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-05-19 10:49:40.841170: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-05-19 10:49:41.899627: W tensorflow/compiler/tf2

In [2]:
#import  data

IMG_SIZE = 224
BATCH_SIZE = 32

train_data = tf.keras.utils.image_dataset_from_directory(
    "data/train",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

test_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/test",
    shuffle = True,
    image_size = (IMG_SIZE, IMG_SIZE),
    batch_size = BATCH_SIZE
)

val_data = tf.keras.utils.image_dataset_from_directory(
    directory = "data/valid",
    shuffle = True,
    image_size = (IMG_SIZE,IMG_SIZE),
    batch_size = BATCH_SIZE
)

data_augmentation_layers = [
    tf.keras.layers.RandomFlip("horizontal"),
    tf.keras.layers.RandomRotation(0.1),
    tf.keras.layers.RandomZoom(0.1),
    tf.keras.layers.RandomContrast(0.1),
]

def  data_augmentation(images):
    for layer in  data_augmentation_layers:
        images = layer(images)
    return images

def preprocess_image(image, label):
    image = tf.cast(image, tf.float32) / 255.0  # Normalize pixel values to [0, 1]
    image = data_augmentation(image)
    return image, label

train_data = train_data.map(
    preprocess_image,
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)

val_data = val_data.map(
    lambda img, label: (tf.cast(img, tf.float32) / 255.0, label),
    num_parallel_calls=tf.data.AUTOTUNE
).prefetch(tf.data.AUTOTUNE)


Found 2339 files belonging to 10 classes.


2024-05-19 10:49:44.107317: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-19 10:49:44.280876: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-19 10:49:44.280935: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-19 10:49:44.283753: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-05-19 10:49:44.283803: I external/local_xla/xla/stream_executor

Found 50 files belonging to 10 classes.
Found 50 files belonging to 10 classes.


In [36]:
IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS =3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def Model(input_shape):

    model = keras.Sequential([
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
       
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Flatten(),
        keras.layers.Dense(64, activation='relu'),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(10, activation='softmax'),
    ])
    
    
    return model

Model(input_shape)

In [37]:
model = Model(input_shape)
model.compile(optimizer = 'adam',
             loss = tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
             metrics=['accuracy'])

In [38]:
history = model.fit(
train_data,
epochs = 12,
batch_size = 32,
validation_data = val_data)

Epoch 1/12
74/74 [==============================] - 7s 53ms/step - loss: 1.9653 - accuracy: 0.2638 - val_loss: 2.8992 - val_accuracy: 0.1200
Epoch 2/12
74/74 [==============================] - 4s 43ms/step - loss: 1.6901 - accuracy: 0.3643 - val_loss: 3.3117 - val_accuracy: 0.1200
Epoch 3/12
74/74 [==============================] - 4s 43ms/step - loss: 1.5685 - accuracy: 0.3985 - val_loss: 2.7658 - val_accuracy: 0.1600
Epoch 4/12
74/74 [==============================] - 4s 43ms/step - loss: 1.4394 - accuracy: 0.4519 - val_loss: 2.4510 - val_accuracy: 0.2200
Epoch 5/12
74/74 [==============================] - 4s 43ms/step - loss: 1.3982 - accuracy: 0.4656 - val_loss: 2.9919 - val_accuracy: 0.2000
Epoch 6/12
74/74 [==============================] - 4s 43ms/step - loss: 1.3691 - accuracy: 0.4664 - val_loss: 2.3878 - val_accuracy: 0.2200
Epoch 7/12
74/74 [==============================] - 4s 42ms/step - loss: 1.3326 - accuracy: 0.4917 - val_loss: 2.4936 - val_accuracy: 0.4000
Epoch 8/12
74

In [4]:
import tensorflow as tf
from tensorflow import keras
import keras_tuner as kt

IMG_SIZE = 224
BATCH_SIZE = 32
CHANNELS = 3
input_shape = (IMG_SIZE, IMG_SIZE, CHANNELS)

def HModel(hp):
    model = keras.Sequential([
        
        keras.layers.Conv2D(32,kernel_size=3,input_shape=input_shape),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
       
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        keras.layers.Conv2D(64,kernel_size=3),
        keras.layers.BatchNormalization(),
        keras.layers.ReLU(),
        keras.layers.MaxPool2D((2,2)),
        
        
        keras.layers.Flatten(),
        keras.layers.Dropout(0.2),
        keras.layers.Dense(units=hp.Int("units", min_value=32, max_value=512, step=32), activation="relu"),
        keras.layers.Dense(10, activation='softmax'),
    ])
    
    learning_rate = hp.Float("lr", min_value=1e-4, max_value=1e-2, sampling="log")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits =False),
        metrics=["accuracy"],
    )
    return model

HModel(kt.HyperParameters())

tuner = kt.RandomSearch(
    hypermodel=HModel,
    objective="val_accuracy",
    max_trials=10,
    executions_per_trial=5,
    overwrite=True,
    directory="./ACML_Project",
    project_name="ACML",
)
tuner.search_space_summary()


tuner.search(train_data, epochs=10, validation_data=val_data)

models = tuner.get_best_models(num_models=2)
best_model = models[0]
best_model.summary()

#Retraining the model with the best hyperparamters
#Top 2
best = tuner.get_best_hyperparameters(5)

#Building model
model = HModel(best[0])


earlystopping = tf.keras.callbacks.EarlyStopping(monitor="val_loss", mode="min", patience=75, restore_best_weights=True)


history = model.fit(
train_data,
epochs = 400,
batch_size = 32,
validation_data = val_data,callbacks=[earlystopping])


Trial 10 Complete [00h 03m 19s]
val_accuracy: 0.5840000033378601

Best val_accuracy So Far: 0.5840000033378601
Total elapsed time: 00h 33m 09s
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 conv2d (Conv2D)             (None, 222, 222, 32)      896       
                                                                 
 max_pooling2d (MaxPooling2  (None, 111, 111, 32)      0         
 D)                                                              
                                                                 
 conv2d_1 (Conv2D)           (None, 109, 109, 64)      18496     
                                                                 
 batch_normalization (Batch  (None, 109, 109, 64)      256       
 Normalization)                                                  
                                                                 
 re_lu (ReLU)                (None, 109, 109,

In [9]:
model.evaluate(val_data, return_dict = True)

2/2 [==============================] - 0s 20ms/step - loss: 0.3274 - accuracy: 0.9400


{'loss': 0.3273526430130005, 'accuracy': 0.9399999976158142}

In [8]:
model.evaluate(test_data, return_dict = True)

2/2 [==============================] - 0s 12ms/step - loss: 2911.4958 - accuracy: 0.2600


{'loss': 2911.495849609375, 'accuracy': 0.25999999046325684}